# ALPHA-MATH · AIMO Progress Prize 3

**Model:** Qwen2.5-Math-7B (open-weight math specialist)  
**Runtime:** Kaggle GPU · **no external API** · **no internet at inference**  
**Loop:** model writes Python → sandbox executes → self-repair → majority vote

Aligned with published kernel: `danielsolo1770/alpha-math`

### Setup before Run All
1. Attach competition data: *AI Mathematical Olympiad - Progress Prize 3*
2. Attach Qwen2.5-Math-7B weights (Kaggle Model e.g. `urvishp80/qwen-2.5-math-7b`)
3. Copy this repo into `/kaggle/working/AlphaMath` (or attach as Dataset)
4. Notebook settings: **GPU on**, **Internet off** for final submission run


In [ ]:
import os, sys
from pathlib import Path

REPO = Path("/kaggle/working/AlphaMath")
if not REPO.exists():
    cands = list(Path("/kaggle/input").glob("**/src/agent.py"))
    if cands:
        REPO = cands[0].parents[1]
print("REPO:", REPO)
sys.path.insert(0, str(REPO))

# Prefer env override, else the path used in the published notebook, else auto-discover
MODEL_PATH = os.environ.get("ALPHAMATH_MODEL_PATH")
DEFAULT_QWEN = Path(
    "/kaggle/input/models/urvishp80/qwen-2.5-math-7b/transformers/default/1"
)
if not MODEL_PATH:
    if DEFAULT_QWEN.exists():
        MODEL_PATH = str(DEFAULT_QWEN)
    else:
        hits = sorted(Path("/kaggle/input").glob("**/config.json"))
        ranked = [
            p.parent
            for p in hits
            if any(k in str(p).lower() for k in ("qwen", "math", "deepseek"))
        ]
        MODEL_PATH = str(ranked[0]) if ranked else None
print("MODEL_PATH:", MODEL_PATH)
assert MODEL_PATH and Path(MODEL_PATH).exists(), (
    "Attach Qwen2.5-Math weights and set MODEL_PATH to the folder with config.json"
)

TEST_CANDIDATES = [
    p for p in Path("/kaggle/input").glob("**/test.csv") if "models" not in p.parts
]
print("test.csv candidates:", TEST_CANDIDATES[:5])


In [ ]:
import torch
print(
    "cuda:",
    torch.cuda.is_available(),
    torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
)


In [ ]:
from src.kaggle_submit import run_submission

test_csv = str(TEST_CANDIDATES[0]) if TEST_CANDIDATES else None
out = run_submission(
    config_path=str(REPO / "configs" / "kaggle.yaml"),
    test_csv=test_csv,
    out_csv="/kaggle/working/submission.csv",
    model_path=MODEL_PATH,
    # limit=3,  # uncomment for a dry run
)
print("Wrote", out)


In [ ]:
import pandas as pd
df = pd.read_csv("/kaggle/working/submission.csv")
display(df.head(20))
print(df.shape)
print(df.columns.tolist())
